<a href="https://colab.research.google.com/github/Mai732/ML-AI/blob/main/assignment_12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
# 2. Define Dataset Paths (Adjust these to match your uploaded folder name)

base_dir = "/content/Vegetable Images"  # Change this if your folder name is different
train_dir = os.path.join(base_dir, "train")
test_dir = os.path.join(base_dir, "test")
val_dir = os.path.join(base_dir, "validation")

# Verify the paths exist
print("Train path exists:", os.path.exists(train_dir))
print("Test path exists:", os.path.exists(test_dir))
print("Validation path exists:", os.path.exists(val_dir))

In [ ]:
# 3. Count Images in Each Class
def count_images_in_classes(dataset_path):
    classes = os.listdir(dataset_path)
    class_counts = {}
    for cls in classes:
        cls_path = os.path.join(dataset_path, cls)
        if os.path.isdir(cls_path):
            class_counts[cls] = len(os.listdir(cls_path))
    return class_counts

train_counts = count_images_in_classes(train_dir)
test_counts = count_images_in_classes(test_dir)
val_counts = count_images_in_classes(val_dir)

print("Train counts:", train_counts)
print("Test counts:", test_counts)
print("Validation counts:", val_counts)

In [ ]:
# 4. Create a Summary DataFrame
df_train = pd.DataFrame(list(train_counts.items()), columns=["Vegetable", "Train Count"])
df_test = pd.DataFrame(list(test_counts.items()), columns=["Vegetable", "Test Count"])
df_val = pd.DataFrame(list(val_counts.items()), columns=["Vegetable", "Validation Count"])

df_summary = df_train.merge(df_test, on="Vegetable").merge(df_val, on="Vegetable")
df_summary = df_summary.sort_values(by="Vegetable")
print(df_summary)

In [ ]:
# 5. Visualize Class Distribution
df_melted = df_summary.melt(id_vars="Vegetable", var_name="Dataset", value_name="Count")

plt.figure(figsize=(14,6))
sns.barplot(data=df_melted, x="Vegetable", y="Count", hue="Dataset")
plt.title("Class Distribution Across Train, Test, Validation", fontsize=14)
plt.xlabel("Vegetable Type")
plt.ylabel("Number of Images")
plt.xticks(rotation=90)
plt.legend()
plt.tight_layout()
plt.show()

print("Is the dataset balanced?")
print("Yes, the dataset is balanced because each vegetable class has approximately the same number of images across train, test, and validation splits.")

In [ ]:
# 6. Image Preprocessing and Augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

In [ ]:
# 7. Load Images Using flow_from_directory
img_size = (224, 224)
batch_size = 32

train_ds = train_datagen.flow_from_directory(
    train_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical",
    subset="training",
    seed=42
)

val_ds = val_datagen.flow_from_directory(
    train_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical",
    subset="validation",
    seed=42
)

print(f"\nNumber of training batches: {len(train_ds)}")
print(f"Number of validation batches: {len(val_ds)}")

In [ ]:
# 8. Build the CNN Model
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(512, activation='relu'),
    Dropout(0.5),
    Dense(15, activation='softmax')
])

In [ ]:
# 9. Compile the Model
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

print("\nExplanation:")
print("Number of convolutional layers: 3")
print("Purpose of pooling layers: Reduce spatial dimensions to decrease computation and prevent overfitting")
print("Purpose of dropout: Randomly disable 50% of neurons during training to prevent overfitting")
print("Why softmax in output layer: Converts final layer outputs to probabilities that sum to 1 for 15-class classification")

In [ ]:
# 10. Train the Model with EarlyStopping
early_stop = EarlyStopping(monitor='val_loss',
                           patience=5,
                           restore_best_weights=True)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[early_stop],
    verbose=1
)

print("\nWhy EarlyStopping is useful:")
print("It monitors validation loss and stops training when it stops improving for 5 epochs.")
print("This prevents overfitting and saves training time by restoring the best model weights.")

In [ ]:
# 11. Evaluate the Model
print("Evaluating on training data...")
train_loss, train_acc = model.evaluate(train_ds, verbose=0)

print("Evaluating on validation data...")
val_loss, val_acc = model.evaluate(val_ds, verbose=0)

print(f"\n{'='*40}")
print(f"Training Accuracy: {train_acc:.4f}")
print(f"Validation Accuracy: {val_acc:.4f}")
print(f"Training Loss: {train_loss:.4f}")
print(f"Validation Loss: {val_loss:.4f}")
print(f"{'='*40}")

print("\nIs there a large difference between training and validation accuracy?")
difference = train_acc - val_acc
if difference > 0.1:
    print(f"Yes, difference of {difference:.4f} indicates overfitting.")
    print("The model is memorizing training data but not generalizing well.")
else:
    print(f"No, difference of only {difference:.4f} means the model generalizes well.")

In [ ]:
# 12. Plot Accuracy and Loss Curves
plt.figure(figsize=(14,5))

# Accuracy plot
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], 'b-', label='Training Accuracy', linewidth=2)
plt.plot(history.history['val_accuracy'], 'r-', label='Validation Accuracy', linewidth=2)
plt.title('Model Accuracy Over Epochs', fontsize=14)
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

# Loss plot
plt.subplot(1,2,2)
plt.plot(history.history['loss'], 'b-', label='Training Loss', linewidth=2)
plt.plot(history.history['val_loss'], 'r-', label='Validation Loss', linewidth=2)
plt.title('Model Loss Over Epochs', fontsize=14)
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nFinal epochs summary:")
print(f"Total epochs run: {len(history.history['accuracy'])}")
print(f"Best validation accuracy: {max(history.history['val_accuracy']):.4f}")
print(f"Best training accuracy: {max(history.history['accuracy']):.4f}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')